# 8 - ROC / AUC Analysis

**Purpose**: ROC curves comparing CIDER Score, TMB, and their combination for predicting binary ICI response. Per-cohort AUC values are embedded directly in the supplement figure.

The combined model's formula is `response ~ log2(CIDER T-cell Score + 1) + log2(TMB per Mb + 1)` (`cider_log2` + `tmb_log2` below) -- log2 for both, so a one-unit change in either predictor means "the raw value doubled" (same convention used for CIDER, TMB, and RNA throughout the paper). The "Overall" (pan-cancer) fit/CV/AUC uses `roc_data_pooled` -- one row per ICI-treated patient -- not the per-cohort `roc_data`, which deliberately keeps a duplicate "ICI-Treated" row per patient so the per-cohort ROC panel can show a pooled facet (see Section 0). The pooled combined model is saved to `figs/tmb_cider_combined_model.rds` and loaded (not refit) by `7_predicted_response_landscape.ipynb`, so the "TMB + CIDER Score" AUC shown in Figure 5B and Figure 7 Panel C is the exact same number -- run this notebook before that one.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/cpi_ready.rds` |
| **Outputs** | `figs/supplement_roc.png` |

In [1]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [2]:
library(ggpubr)
library(pROC)
library(caret)
library(patchwork)

Type 'citation("pROC")' for a citation.




Attaching package: ‘pROC’




The following objects are masked from ‘package:stats’:

    cov, smooth, var




# 0 - Read data

In [ ]:
# T-cell group only; drop samples with no documented response.
# cider_log2 = log2(cider_score_depth_adj + 1)
# tmb_log2   = log2(tmbPerMb + 1)  — tmbPerMb is the raw value here (not
# pre-transformed upstream -- see 0_prepare/3_meta.ipynb); log2 is applied
# at this point of use instead, same convention as CIDER.
roc_data <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  dplyr::filter(gp == "T-cell", !is.na(response)) |>
  dplyr::select(sampleId, cohort_broad, cohort_broad_go, response,
         cider_score_adj, cider_score_depth_adj, tmbPerMb) |>
  dplyr::mutate(
    response   = as.integer(response),
    cider_log2 = log2(cider_score_depth_adj + 1),
    tmb_log2   = log2(tmbPerMb + 1)
  )

# Pooled, one-row-per-patient version of roc_data, for every "Overall"
# (pan-cancer) computation below -- the CV fit and the single canonical
# combined-model AUC shared with 7_predicted_response_landscape.ipynb
# (Figure 7). cpi_ready.rds carries one row per (sampleId, gp) *and* a
# duplicated copy of every ICI-treated patient with cohort_broad overwritten
# to "ICI-Treated" (4_base.ipynb). roc_data above deliberately keeps both
# copies so the per-cohort ROC curves (Section 2) can show a pooled
# "ICI-Treated" facet alongside each real cancer type (same mechanism as
# Figure 3 / 17_response_by_read_confidence.ipynb) -- each per-cohort fit
# only ever sees ONE row per patient (whichever facet it's grouped into), so
# that part was never affected. But feeding BOTH copies of the same patient
# into one *pooled* fit (no cohort grouping) doubles N and lets a patient's
# two identical rows land in different CV folds -- training-set leakage into
# the "held-out" fold, inflating the CV AUC. roc_data_pooled removes exactly
# that duplicate, same convention as 7_predicted_response_landscape.ipynb's
# own `train_df`.
roc_data_pooled <- roc_data |>
  dplyr::filter(cohort_broad != "ICI-Treated") |>
  dplyr::distinct(sampleId, .keep_all = TRUE)


In [ ]:
# Cross-validated AUC for the combined model (10-fold × 10 repeats).
# Predictors: log2(CIDER T-cell Score + 1) and log2(TMB per Mb + 1).
# Addresses optimism bias from training and evaluating on the same data.

set.seed(42)

cv_data <- roc_data_pooled |>
  filter(!is.na(response), !is.na(cider_log2), !is.na(tmb_log2))

cv_ctrl <- trainControl(
  method          = "repeatedcv",
  number          = 10,
  repeats         = 10,
  classProbs      = TRUE,
  summaryFunction = twoClassSummary,
  savePredictions = "final"
)

# caret requires factor outcome with valid R names
cv_data <- cv_data |>
  mutate(response_fct = factor(ifelse(response == 1, "Responder", "NonResponder"),
                                levels = c("Responder", "NonResponder")))

cv_fit_combined <- train(
  response_fct ~ cider_log2 + tmb_log2,
  data      = cv_data,
  method    = "glm",
  family    = "binomial",
  metric    = "ROC",
  trControl = cv_ctrl
)

cv_fit_cider <- train(
  response_fct ~ cider_log2,
  data      = cv_data,
  method    = "glm",
  family    = "binomial",
  metric    = "ROC",
  trControl = cv_ctrl
)

cv_fit_tmb <- train(
  response_fct ~ tmb_log2,
  data      = cv_data,
  method    = "glm",
  family    = "binomial",
  metric    = "ROC",
  trControl = cv_ctrl
)

# Means: signif(..., 2) — 2 significant figures
# SDs:   round(..., 2)  — 2 decimal places

cv_combined_mean <- signif(cv_fit_combined$results$ROC,   2)
cv_combined_sd   <- round(cv_fit_combined$results$ROCSD,  2)
cv_cider_mean    <- signif(cv_fit_cider$results$ROC,      2)
cv_cider_sd      <- round(cv_fit_cider$results$ROCSD,     2)
cv_tmb_mean      <- signif(cv_fit_tmb$results$ROC,        2)
cv_tmb_sd        <- round(cv_fit_tmb$results$ROCSD,       2)

message("CV AUC (10-fold x 10 repeats):")
message("  TMB + CIDER Score : ", cv_combined_mean, " +/- ", cv_combined_sd)
message("  CIDER Score       : ", cv_cider_mean,    " +/- ", cv_cider_sd)
message("  TMB               : ", cv_tmb_mean,      " +/- ", cv_tmb_sd)


# 1 - ROC helper functions

In [ ]:
# Three predictors:
#   "CIDER Score"        — log2(cider_score_depth_adj + 1)
#   "TMB"                — log2(tmbPerMb + 1)
#   "TMB + CIDER Score"  — combined logistic model (T-cell + TMB)
#
# DeLong tests:
#   combined vs CIDER alone  (does TMB add value?)
#   combined vs TMB alone    (does CIDER add value?)

compute_rocs <- function(df) {
  df <- df |> dplyr::filter(!is.na(response), !is.na(cider_log2), !is.na(tmb_log2))
  if (nrow(df) < 10) return(NULL)

  combined_prob <- tryCatch(
    predict(
      glm(response ~ cider_log2 + tmb_log2, data = df, family = "binomial"),
      newdata = df, type = "response"
    ),
    error = function(e) rep(NA_real_, nrow(df))
  )

  predictors <- list(
    "CIDER Score"       = df$cider_log2,
    "TMB"               = df$tmb_log2,
    "TMB + CIDER Score" = combined_prob
  )

  curves <- purrr::map(names(predictors), function(nm) {
    vals <- predictors[[nm]]
    if (all(is.na(vals))) return(NULL)
    pROC::roc(df$response, vals, quiet = TRUE)
  }) |> setNames(names(predictors))

  fmt_p <- function(p) fmt_pval(p)

  delong_cider <- tryCatch(
    fmt_p(pROC::roc.test(curves[["CIDER Score"]], curves[["TMB + CIDER Score"]],
                         method = "delong")$p.value), error = function(e) NA_character_)
  delong_tmb <- tryCatch(
    fmt_p(pROC::roc.test(curves[["TMB"]], curves[["TMB + CIDER Score"]],
                         method = "delong")$p.value), error = function(e) NA_character_)

  purrr::map_dfr(names(curves), function(nm) {
    if (is.null(curves[[nm]])) return(NULL)
    pROC::coords(curves[[nm]], "all",
                 ret = c("threshold", "sensitivity", "specificity"),
                 transpose = FALSE) |>
      dplyr::mutate(
        predictor       = nm,
        auc             = as.numeric(pROC::auc(curves[[nm]])),
        delong_vs_cider = delong_cider,
        delong_vs_tmb   = delong_tmb
      )
  })
}

auc_label <- function(nm, auc_val) {
  paste0(nm, " (AUC = ", round(auc_val, 2), ")")
}

# "TMB + CIDER Score" is the headline combined model shown in Figure 5B, so
# it gets COL_POSITIVE (orange) -- the colour reserved project-wide for the
# headline "CIDER + TMB" model in ROC/AUC model-comparison panels (see
# helpers/themes.r). "CIDER Score" alone takes COL_ACCENT (blue) instead --
# swapped from this palette's otherwise fixed POSITIVE/NEGATIVE/MID/ACCENT
# cycle so the two never collide. "Full (Regularized)" (the larger
# multi-feature model from 12_xgboost_model.ipynb, only added to the compact
# Figure 5B panel when that notebook's ROC curve is available) keeps the
# sage green slot.
roc_palette <- c(
  "CIDER Score"          = COL_ACCENT,
  "TMB"                  = COL_NEGATIVE,
  "TMB + CIDER Score"    = COL_POSITIVE,
  "Full (Regularized)"   = COL_MID
)


In [ ]:
cohort_roc_curves <- roc_data |>
  group_by(cohort_broad, cohort_broad_go) |>
  group_modify(~ compute_rocs(.x)) |>
  ungroup()


# 2 - Per-cohort ROC curves

In [ ]:
# Overall (pan-cancer) ROC — all ICI-treated patients pooled, one row per
# patient (roc_data_pooled drops the "ICI-Treated" duplicate -- see Section 0
# -- so this fit/evaluation can't double-count a patient or leak across CV
# folds the way the undeduplicated roc_data would).
overall_roc_curves <- compute_rocs(roc_data_pooled)

overall_auc <- overall_roc_curves |>
  distinct(predictor, auc) |>
  mutate(predictor = factor(predictor, levels = names(roc_palette))) |>
  arrange(predictor)

message("Overall AUC (in-sample, deduplicated pooled population):")
print(overall_auc)

# ── Canonical "TMB + CIDER Score" model ───────────────────────────────────────
# Same formula/data/family compute_rocs() already fits internally for that
# predictor (response ~ cider_log2 + tmb_log2, binomial, on roc_data_pooled) --
# refit explicitly here (not extracted from compute_rocs()'s internals) so the
# model object itself can be saved and reused by
# 7_predicted_response_landscape.ipynb instead of being independently refit
# there (same model/training population feeds its predictions). The AUC
# number both figures DISPLAY, though, is the CV mean +/- SD computed above
# (cv_combined_mean/cv_combined_sd) -- saved alongside the model so Figure 7
# Panel C's title shows the same "AUC x.xx +/- y.yy" text as Figure 5B's
# "TMB + CIDER Score" label, not an in-sample point estimate.
tmb_cider_model         <- glm(response ~ cider_log2 + tmb_log2, data = cv_data, family = binomial())
tmb_cider_auc_in_sample <- round(overall_auc$auc[overall_auc$predictor == "TMB + CIDER Score"], 3)

saveRDS(
  list(
    model         = tmb_cider_model,
    data          = cv_data,
    auc_in_sample = tmb_cider_auc_in_sample,
    cv_auc_mean   = cv_combined_mean,
    cv_auc_sd     = cv_combined_sd
  ),
  paste0(CIDER_PROJ_DIR, "figs/tmb_cider_combined_model.rds")
)
message("Saved: tmb_cider_combined_model.rds (canonical TMB + CIDER Score model; CV AUC = ",
        cv_combined_mean, " ± ", cv_combined_sd, ")")

ORIG_PREDICTORS    <- c("CIDER Score", "TMB", "TMB + CIDER Score")
OVERALL_PREDICTORS <- c("CIDER Score", "TMB", "TMB + CIDER Score", "Full (Regularized)")

cohort_auc_labels <- cohort_roc_curves |>
  dplyr::filter(predictor %in% ORIG_PREDICTORS) |>
  distinct(cohort_broad, cohort_broad_go, predictor, auc,
           delong_vs_cider, delong_vs_tmb) |>
  mutate(
    predictor = factor(predictor, levels = ORIG_PREDICTORS),
    x_pos     = 0.97,
    y_pos     = c(0.18, 0.10, 0.02)[as.integer(predictor)]
  )

delong_labels <- cohort_auc_labels |>
  filter(predictor == "TMB + CIDER Score") |>
  distinct(cohort_broad_go, delong_vs_cider, delong_vs_tmb) |>
  mutate(
    label = paste0(
      "vs CIDER: ", delong_vs_cider, "\n",
      "vs TMB: ",   delong_vs_tmb
    )
  )

supplement_roc <- cohort_roc_curves |>
  dplyr::filter(predictor %in% ORIG_PREDICTORS) |>
  mutate(predictor = factor(predictor, levels = ORIG_PREDICTORS)) |>
  ggplot(aes(x = 1 - specificity, y = sensitivity, colour = predictor)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = COL_NEGATIVE) +
  geom_line(linewidth = 0.7) +
  geom_text(
    data        = cohort_auc_labels,
    aes(x = x_pos, y = y_pos,
        label   = paste0(predictor, ": ", round(auc, 2)),
        colour  = predictor),
    size         = 2.4, hjust = 1, inherit.aes = FALSE, show.legend = FALSE
  ) +
  geom_text(
    data       = delong_labels,
    aes(x = 0.00, y = 0.97, label = label),
    colour     = COL_INK,
    size       = 2.2, hjust  = 0, vjust = 1.2, inherit.aes = FALSE
  ) +
  scale_colour_manual(values = roc_palette) +
  go_theme +
  coord_equal() +
  facet_wrap(~cohort_broad_go, nrow = 2) +
  labs(
    x        = "1 - Specificity",
    y        = "Sensitivity",
    colour   = NULL,
    title    = "ROC Curves by Broad Cohort — DeLong test: TMB + CIDER vs individual predictors",
    subtitle = paste0(
      "Overall AUC  |  ",
      "TMB + CIDER Score: ", round(overall_auc$auc[overall_auc$predictor == "TMB + CIDER Score"], 2),
      " (CV: ", cv_combined_mean, " ± ", cv_combined_sd, ")",
      "   TMB: ",            round(overall_auc$auc[overall_auc$predictor == "TMB"], 2),
      " (CV: ", cv_tmb_mean, " ± ", cv_tmb_sd, ")",
      "   CIDER Score: ",    round(overall_auc$auc[overall_auc$predictor == "CIDER Score"], 2),
      " (CV: ", cv_cider_mean, " ± ", cv_cider_sd, ")"
    )
  )

saveRDS(supplement_roc, paste0(CIDER_PROJ_DIR, "figs/supplement_roc.rds"))
set_plot_dims(14, 9); supplement_roc

# ── Overall DeLong p-values (for Figure 4B annotation) ────────────────────────────────────────────
overall_delong_cider <- overall_roc_curves |>
  filter(predictor == "TMB + CIDER Score") |>
  dplyr::pull(delong_vs_cider) |>
  dplyr::first()
overall_delong_tmb <- overall_roc_curves |>
  filter(predictor == "TMB + CIDER Score") |>
  dplyr::pull(delong_vs_tmb) |>
  dplyr::first()

# ── Overall pan-cancer ROC — compact single panel for Figure 4B ───────────────────────
# Labels show CV AUC only (cross-validated; not the in-sample point estimate).
# DeLong test (combined vs each individual predictor) annotated bottom-left.
# Load Full model ROC curve from 12_xgboost_model.ipynb (if available)
lr_full_roc_file <- paste0(CIDER_PROJ_DIR, "figs/lr_full_roc_curve.rds")
lr_full_roc_data <- if (file.exists(lr_full_roc_file)) readRDS(lr_full_roc_file) else NULL

# Combine overall curves with full model curve
overall_roc_all <- dplyr::bind_rows(
  overall_roc_curves |> dplyr::select(predictor, sensitivity, specificity, auc),
  if (!is.null(lr_full_roc_data))
    lr_full_roc_data |> dplyr::select(predictor, sensitivity, specificity, auc)
  else NULL
)

overall_auc_labels <- dplyr::bind_rows(
  tibble(
    predictor = ORIG_PREDICTORS,
    cv_mean   = c(cv_cider_mean, cv_tmb_mean, cv_combined_mean),
    cv_sd     = c(cv_cider_sd,   cv_tmb_sd,   cv_combined_sd),
    x_pos     = 0.97,
    y_pos     = c(0.28, 0.20, 0.12)
  ),
  if (!is.null(lr_full_roc_data)) tibble(
    predictor = "Full (Regularized)",
    cv_mean   = unique(lr_full_roc_data$auc_mean),
    cv_sd     = unique(lr_full_roc_data$auc_sd),
    x_pos     = 0.97,
    y_pos     = 0.04
  ) else NULL
) |>
  dplyr::mutate(
    predictor = factor(predictor, levels = OVERALL_PREDICTORS),
    # Same CV-mean +/- SD format for every predictor, including "TMB + CIDER
    # Score" -- Figure 7 Panel C's "Model AUC" title is built from this same
    # cv_combined_mean/cv_combined_sd (saved in tmb_cider_combined_model.rds),
    # so the two figures display identical "AUC x.xx ± y.yy" text.
    label = paste0(predictor, ": AUC ", cv_mean, " ± ", cv_sd)
  )

overall_roc_plot <- overall_roc_all |>
  dplyr::filter(predictor %in% OVERALL_PREDICTORS) |>
  mutate(predictor = factor(predictor, levels = OVERALL_PREDICTORS)) |>
  ggplot(aes(x = 1 - specificity, y = sensitivity, colour = predictor)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = COL_NEGATIVE) +
  geom_line(linewidth = 0.9) +
  geom_text(
    data        = overall_auc_labels,
    aes(x = x_pos, y = y_pos, label = label, colour = predictor),
    size        = 2.8, hjust = 1, inherit.aes = FALSE, show.legend = FALSE
  ) +
  annotate("text",
    x = 0.00, y = 0.99,
    label = paste0(
      "DeLong: combined vs CIDER  ", overall_delong_cider, "\n",
      "DeLong: combined vs TMB    ", overall_delong_tmb
    ),
    hjust = 0, vjust = 1, size = 2.5, colour = COL_INK
  ) +
  scale_colour_manual(values = roc_palette, name = NULL) +
  go_theme +
  coord_equal() +
  theme(legend.position = "none", plot.margin = margin(t = 2, r = 4, b = 2, l = 4)) +
  labs(
    title    = "Response Prediction (ROC)",
    x        = "1 - Specificity",
    y        = "Sensitivity"
  )

set_plot_dims(5, 5); overall_roc_plot
saveRDS(overall_roc_plot, paste0(CIDER_PROJ_DIR, "figs/overall_roc.rds"))
